In [ ]:
"""Imports and plotting setup"""

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['ps.useafm'] = True
plt.rcParams['pdf.use14corefonts'] = True
from torch.autograd import grad
from scipy.stats.qmc import LatinHypercube
from collections import defaultdict
import os
import time


In [ ]:
"""Runtime setup"""

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(1234)
torch.manual_seed(1234)
print(f"Using device: {device}")


In [ ]:
"""Geometry and boundary definitions"""


A = np.array([0.0, 0.25])
B = np.array([0.4, 0.3])
C = np.array([0.3, 0.0])
D = np.array([0.5, 0.0])
E = np.array([0.6, 0.35])
F = np.array([1.0, 0.5])
G = np.array([1.0, 0.75])
H = np.array([0.0, 0.5])

y_shape_vertices = np.array([H, A, B, C, D, E, F, G])


print(f"  A={A}, B={B}, C={C}, D={D}")
print(f"  E={E}, F={F}, G={G}, H={H}")


In [ ]:
"""Geometry and boundary definitions"""



def quadratic_bezier(p0, p1, p2, t):
    t = np.atleast_1d(t)
    p0 = np.atleast_1d(p0)
    p1 = np.atleast_1d(p1)
    p2 = np.atleast_1d(p2)

    if t.ndim == 1 and p0.ndim == 1:
        return (1 - t[0]) ** 2 * p0 + 2 * (1 - t[0]) * t[0] * p1 + t[0] ** 2 * p2
    else:
        t = t.reshape(-1, 1)
        return (1 - t) ** 2 * p0 + 2 * (1 - t) * t * p1 + t ** 2 * p2

def quadratic_bezier_derivative(p0, p1, p2, t):
    return 2 * (1 - t) * (p1 - p0) + 2 * t * (p2 - p1)

def cubic_bezier(p0, p1, p2, p3, t):
    t = np.atleast_1d(t)
    if t.ndim == 1 and len(t) == 1:
        t = t[0]
        return (1-t)**3 * p0 + 3*(1-t)**2*t * p1 + 3*(1-t)*t**2 * p2 + t**3 * p3
    else:
        t = t.reshape(-1, 1)
        return (1-t)**3 * p0 + 3*(1-t)**2*t * p1 + 3*(1-t)*t**2 * p2 + t**3 * p3

def cubic_bezier_derivative(p0, p1, p2, p3, t):

    return 3*(1-t)**2*(p1-p0) + 6*(1-t)*t*(p2-p1) + 3*t**2*(p3-p2)

AB_ctrl = np.array([0.2, 0.12])

EF_ctrl = np.array([0.8, 0.58])

GH_ctrl1 = np.array([0.7, 0.90])
GH_ctrl2 = np.array([0.3, 0.30])

print("curved boundarycontrol points定义completed:")
print(f"  ABcontrol points: {AB_ctrl} (downward curve)")
print(f"  EFcontrol points: {EF_ctrl} (upward curve)")
print(f"  GHcontrol points1: {GH_ctrl1}, control points2: {GH_ctrl2} (S-shaped curve)")

curve_segments = {
    'AB': {'p0': A, 'ctrl': AB_ctrl, 'p2': B, 'type': 'quadratic'},
    'EF': {'p0': E, 'ctrl': EF_ctrl, 'p2': F, 'type': 'quadratic'},
    'GH': {'p0': G, 'ctrl1': GH_ctrl1, 'ctrl2': GH_ctrl2, 'p3': H, 'type': 'cubic'},
}

line_segments = {
    'HA': {'start': H, 'end': A},
    'BC': {'start': B, 'end': C},
    'CD': {'start': C, 'end': D},
    'DE': {'start': D, 'end': E},
    'FG': {'start': F, 'end': G},
}

interface_segments = {
    'AB': {'type': 'quadratic', 'p0': A, 'ctrl': AB_ctrl, 'p2': B, 'outward': 'right'},
    'BC': {'type': 'line', 'start': B, 'end': C, 'outward': 'right'},
    'DE': {'type': 'line', 'start': D, 'end': E, 'outward': 'right'},
    'EF': {'type': 'quadratic', 'p0': E, 'ctrl': EF_ctrl, 'p2': F, 'outward': 'right'},
    'GH': {'type': 'cubic', 'p0': G, 'ctrl1': GH_ctrl1, 'ctrl2': GH_ctrl2, 'p3': H, 'outward': 'right'},
}

velocity_bc_segments = {
    'HA': {'start': H, 'end': A, 'u': (0.5, 0)},
    'CD': {'start': C, 'end': D, 'u': (0, 0.5)},
    'FG': {'start': F, 'end': G, 'u': (1.0, 0)},
}

def compute_unit_normal_and_tangent(p1, p2, outward='left'):
    direction = p2 - p1
    length = np.linalg.norm(direction)
    tangent = direction / length
    if outward == 'left':
        normal = np.array([-tangent[1], tangent[0]])
    else:
        normal = np.array([tangent[1], -tangent[0]])
    return normal, tangent

def get_curve_point(segment_name, t):
    if segment_name in curve_segments:
        seg = curve_segments[segment_name]
        if seg.get('type') == 'cubic':
            return cubic_bezier(seg['p0'], seg['ctrl1'], seg['ctrl2'], seg['p3'], t)
        else:
            return quadratic_bezier(seg['p0'], seg['ctrl'], seg['p2'], t)
    elif segment_name in line_segments:
        seg = line_segments[segment_name]
        return (1 - t) * seg['start'] + t * seg['end']
    else:
        raise ValueError(f"Unknown segment: {segment_name}")

def get_curve_tangent_and_normal(segment_name, t, outward='right'):
    if segment_name in curve_segments:
        seg = curve_segments[segment_name]
        if seg.get('type') == 'cubic':
            tangent_raw = cubic_bezier_derivative(seg['p0'], seg['ctrl1'], seg['ctrl2'], seg['p3'], t)
        else:
            tangent_raw = quadratic_bezier_derivative(seg['p0'], seg['ctrl'], seg['p2'], t)
    elif segment_name in line_segments:
        seg = line_segments[segment_name]
        tangent_raw = seg['end'] - seg['start']
    else:
        raise ValueError(f"Unknown segment: {segment_name}")

    length = np.linalg.norm(tangent_raw)
    if length > 1e-10:
        tangent = tangent_raw / length
    else:
        tangent = np.array([1.0, 0.0])

    if outward == 'right':
        normal = np.array([tangent[1], -tangent[0]])
    else:
        normal = np.array([-tangent[1], tangent[0]])

    return tangent, normal

In [ ]:
"""Geometry helper functions"""


def point_in_polygon(x, y, polygon):
    n = len(polygon)
    inside = False

    j = n - 1
    for i in range(n):
        xi, yi = polygon[i]
        xj, yj = polygon[j]

        if ((yi > y) != (yj > y)) and (x < (xj - xi) * (y - yi) / (yj - yi + 1e-12) + xi):
            inside = not inside
        j = i

    return inside

def points_in_polygon(points, polygon):
    results = np.zeros(len(points), dtype=bool)
    for i, (x, y) in enumerate(points):
        results[i] = point_in_polygon(x, y, polygon)
    return results

def generate_y_shape_boundary_polygon(n_points_per_segment=50):
    boundary_points = []

    t_vals = np.linspace(0, 1, n_points_per_segment, endpoint=False)
    for t in t_vals:
        point = (1 - t) * H + t * A
        boundary_points.append(point)

    for t in t_vals:
        point = get_curve_point('AB', t)
        boundary_points.append(point)

    for t in t_vals:
        point = (1 - t) * B + t * C
        boundary_points.append(point)

    for t in t_vals:
        point = (1 - t) * C + t * D
        boundary_points.append(point)

    for t in t_vals:
        point = (1 - t) * D + t * E
        boundary_points.append(point)

    for t in t_vals:
        point = get_curve_point('EF', t)
        boundary_points.append(point)

    for t in t_vals:
        point = (1 - t) * F + t * G
        boundary_points.append(point)

    for t in t_vals:
        point = get_curve_point('GH', t)
        boundary_points.append(point)

    return np.array(boundary_points)

y_shape_curved_polygon = generate_y_shape_boundary_polygon(n_points_per_segment=100)

def point_in_y_shape(x, y):

    return point_in_polygon(x, y, y_shape_curved_polygon)

def points_in_y_shape(points):

    if len(points.shape) == 1:
        return point_in_y_shape(points[0], points[1])
    return points_in_polygon(points[:, :2], y_shape_curved_polygon)


def get_y_shape_bounds():

    min_x = y_shape_vertices[:, 0].min()
    max_x = y_shape_vertices[:, 0].max()
    min_y = y_shape_vertices[:, 1].min()
    max_y = y_shape_vertices[:, 1].max()
    return min_x, min_y, max_x, max_y


In [ ]:
"""Sampling utilities"""

def sample_in_y_shape(n_points, t_range=(0, 0.5)):

    min_x, min_y, max_x, max_y = get_y_shape_bounds()
    points = []
    attempts = 0
    max_attempts = n_points * 100
    while len(points) < n_points and attempts < max_attempts:
        x = np.random.uniform(min_x, max_x)
        y = np.random.uniform(min_y, max_y)
        if point_in_y_shape(x, y):
            t = np.random.uniform(t_range[0], t_range[1])
            points.append([x, y, t])
        attempts += 1
    return np.array(points)

def sample_in_porous_media(n_points, t_range=(0, 0.5)):

    points = []
    attempts = 0
    max_attempts = n_points * 100
    while len(points) < n_points and attempts < max_attempts:
        x = np.random.uniform(0, 1)
        y = np.random.uniform(0, 1)
        if not point_in_y_shape(x, y):
            t = np.random.uniform(t_range[0], t_range[1])
            points.append([x, y, t])
        attempts += 1
    return np.array(points)

def sample_lhs_in_y_shape(n_points, t_range=(0, 0.5)):

    min_x, min_y, max_x, max_y = get_y_shape_bounds()
    sampler = LatinHypercube(d=3)
    samples = sampler.random(n_points * 5)
    samples[:, 0] = samples[:, 0] * (max_x - min_x) + min_x
    samples[:, 1] = samples[:, 1] * (max_y - min_y) + min_y
    samples[:, 2] = samples[:, 2] * (t_range[1] - t_range[0]) + t_range[0]
    valid_points = []
    for p in samples:
        if point_in_y_shape(p[0], p[1]):
            valid_points.append(p)
        if len(valid_points) >= n_points:
            break
    return np.array(valid_points[:n_points])

def sample_lhs_in_porous_media(n_points, t_range=(0, 0.5)):

    sampler = LatinHypercube(d=3)
    samples = sampler.random(n_points * 5)
    samples[:, 2] = samples[:, 2] * (t_range[1] - t_range[0]) + t_range[0]
    valid_points = []
    for p in samples:
        if not point_in_y_shape(p[0], p[1]):
            valid_points.append(p)
        if len(valid_points) >= n_points:
            break
    return np.array(valid_points[:n_points])


In [ ]:
"""Reward functions"""

def calculate_reward(current_losses, previous_losses, initial_losses):
    if previous_losses is None:
        return 0.0
    total_current = sum(current_losses)
    total_previous = sum(previous_losses)
    if total_previous == 0:
        return 0.0
    improvement = (total_previous - total_current) / total_previous
    if improvement > 0.08:
        base_reward = 2.5 * improvement
    elif improvement > 0.04:
        base_reward = 2.2 * improvement
    elif improvement > 0.02:
        base_reward = 2.0 * improvement
    elif improvement > 0.01:
        base_reward = 1.8 * improvement
    elif improvement > -0.01:
        base_reward = 0.0
    else:
        base_reward = -2.0 * abs(improvement)
    if total_current < 0.05:
        absolute_reward = 1.0
    elif total_current < 0.1:
        absolute_reward = 0.8
    elif total_current < 0.5:
        absolute_reward = 0.5
    elif total_current < 1.0:
        absolute_reward = 0.2
    elif total_current > 20.0:
        absolute_reward = -0.3
    elif total_current > 50.0:
        absolute_reward = -0.5
    else:
        absolute_reward = 0.0
    balance_reward = 0.0
    if len(current_losses) > 1:
        max_loss = max(current_losses)
        min_loss = min(current_losses)
        if max_loss > 0:
            balance_ratio = min_loss / max_loss
            if balance_ratio > 0.1:
                balance_reward = 0.3
            elif balance_ratio < 0.01:
                balance_reward = -0.3
    stability_reward = 0.0
    if abs(improvement) < 0.005:
        if total_current < 1.0:
            stability_reward = 0.2
    total_reward = base_reward + absolute_reward + balance_reward + stability_reward
    return max(min(total_reward, 3.0), -3.0)

def calculate_arch_reward(pre_losses, post_losses, total_params, action_type=None,
                         current_depth=None, current_units=None):
    pre_total = sum(pre_losses)
    post_total = sum(post_losses)
    if pre_total == 0:
        return 0.0
    improvement = (pre_total - post_total) / pre_total
    if improvement > 0.10:
        base_reward = 2.5 * improvement
    elif improvement > 0.06:
        base_reward = 2.2 * improvement
    elif improvement > 0.03:
        base_reward = 2.0 * improvement
    elif improvement > 0.01:
        base_reward = 1.8 * improvement
    elif improvement > -0.02:
        base_reward = 0.0
    else:
        base_reward = -2.2 * abs(improvement)
    if post_total < 0.05:
        absolute_reward = 1.0
    elif post_total < 0.1:
        absolute_reward = 0.8
    elif post_total < 0.5:
        absolute_reward = 0.5
    elif post_total < 1.0:
        absolute_reward = 0.2
    elif post_total > 20.0:
        absolute_reward = -0.3
    elif post_total > 50.0:
        absolute_reward = -0.5
    else:
        absolute_reward = 0.0
    param_reward = 0.0
    if total_params > 500000:
        param_reward = -0.5
    elif total_params > 300000:
        param_reward = -0.3
    elif total_params < 100000:
        param_reward = 0.2
    total_reward = base_reward + absolute_reward + param_reward
    return max(min(total_reward, 3.0), -3.0)


In [ ]:
"""RL controller"""

class RLController:
    def __init__(self, n_actions=24, state_dim=25, hidden_dim=64,
                 min_units=5, max_units=256, min_depth=2, max_depth=10):
        self.policy_net = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, n_actions),
            torch.nn.Softmax(dim=-1)
        ).to(device)
        self.optimizer = torch.optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.gamma = 0.99
        self.min_units = min_units
        self.max_units = max_units
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.min_base_points = 1000
        self.max_base_points = 10000
        self.max_refined_points = 10000
        self.episode_rewards = []
        self.episode_log_probs = []
        self.action_stats = {'weight': 0, 'arch': 0, 'sampling': 0, 'total': 0}
        self.arch_action_stats = {
            'increase_depth': 0, 'increase_width': 0,
            'decrease_depth': 0, 'decrease_width': 0,
            'total_arch': 0, 'total_decisions': 0
        }

    def get_enhanced_state(self, losses, depth, units, loss_history,
                      darcy_base_points, ns_base_points,
                      darcy_refined_points, ns_refined_points,
                      historical_best_loss=None,
                      individual_losses=None):
        state = []

        state.extend(losses)

        state.append(depth / 10.0)
        state.append(units / 256.0)

        training_stability = 0.0
        if len(loss_history) > 5:
            recent_losses = loss_history[-5:]
            loss_std = np.std(recent_losses)
            if loss_std < 0.001:
                training_stability = 1.0
            elif loss_std > 0.1:
                training_stability = -1.0
        state.append(training_stability)

        improvement_trend = 0.0
        if len(loss_history) > 10:
            recent_10 = loss_history[-10:]
            improvement_trend = (recent_10[0] - recent_10[-1]) / (recent_10[0] + 1e-9)
        state.append(improvement_trend)

        medium_term_improvement = 0.0
        if len(loss_history) >= 50:
            recent_50 = loss_history[-50:]
            medium_term_improvement = (recent_50[0] - recent_50[-1]) / (recent_50[0] + 1e-9)
        state.append(medium_term_improvement)

        stagnation_indicator = 0.0
        if len(loss_history) >= 50:
            recent_50 = loss_history[-50:]
            improvement_50 = (recent_50[0] - recent_50[-1]) / (recent_50[0] + 1e-9)
            if abs(improvement_50) < 0.02:
                stagnation_indicator = 1.0
        state.append(stagnation_indicator)

        state.append(darcy_base_points / self.max_base_points)
        state.append(ns_base_points / self.max_base_points)
        state.append(darcy_refined_points / self.max_refined_points)
        state.append(ns_refined_points / self.max_refined_points)

        if historical_best_loss is not None and historical_best_loss > 0:
            current_total = sum(losses)

            loss_ratio = current_total / (historical_best_loss + 1e-9)

            normalized_ratio = min(loss_ratio, 10.0) / 10.0
            state.append(normalized_ratio)
        else:
            state.append(0.5)

        if individual_losses is not None and len(individual_losses) == 7:
            total_loss = sum(individual_losses) + 1e-9
            for loss in individual_losses:
                state.append(loss / total_loss)
        else:

            state.extend([1.0/7.0] * 7)

        return torch.FloatTensor(state).to(device).unsqueeze(0)

    def get_valid_actions(self, depth, units, darcy_base_points, ns_base_points,
                         darcy_refined_points, ns_refined_points):
        valid = [1.0] * 24
        if depth >= self.max_depth:
            valid[14] = 0.0
        if depth <= self.min_depth:
            valid[16] = 0.0
        if units >= self.max_units:
            valid[15] = 0.0
        if units <= self.min_units:
            valid[17] = 0.0
        if darcy_refined_points >= self.max_refined_points:
            valid[18] = 0.0
        if darcy_base_points >= self.max_base_points:
            valid[19] = 0.0
        if darcy_base_points <= self.min_base_points:
            valid[20] = 0.0
        if ns_refined_points >= self.max_refined_points:
            valid[21] = 0.0
        if ns_base_points >= self.max_base_points:
            valid[22] = 0.0
        if ns_base_points <= self.min_base_points:
            valid[23] = 0.0
        return valid

    def select_action(self, state):
        if not isinstance(state, torch.Tensor):
            state_tensor = torch.FloatTensor(state).to(device)
        else:
            state_tensor = state
        probs = self.policy_net(state_tensor)
        state_values = state_tensor.squeeze(0)

        depth = int(state_values[7].item() * 10)
        units = int(state_values[8].item() * 256)

        darcy_base = int(state_values[13].item() * self.max_base_points)
        ns_base = int(state_values[14].item() * self.max_base_points)
        darcy_refined = int(state_values[15].item() * self.max_refined_points)
        ns_refined = int(state_values[16].item() * self.max_refined_points)

        valid_actions = self.get_valid_actions(depth, units, darcy_base, ns_base,
                                               darcy_refined, ns_refined)
        masked_probs = probs * torch.FloatTensor(valid_actions).to(device)
        masked_probs = masked_probs / (masked_probs.sum() + 1e-9)
        self.action_stats['total'] += 1
        action_dist = torch.distributions.Categorical(masked_probs)
        action = action_dist.sample()
        action_id = action.item()
        if action_id < 14:
            self.action_stats['weight'] += 1
        elif action_id < 18:
            self.action_stats['arch'] += 1
            self.arch_action_stats['total_decisions'] += 1
            self.arch_action_stats['total_arch'] += 1
            action_names = ['increase_depth', 'increase_width', 'decrease_depth', 'decrease_width']
            action_name = action_names[action_id - 14]
            self.arch_action_stats[action_name] += 1
            print(f" architecture adjustment: {action_name}")
        else:
            self.action_stats['sampling'] += 1
            sampling_names = {18: "Darcyrefinedsampling", 19: "Darcybase+20%", 20: "Darcybase-20%",
                            21: "NSrefinedsampling", 22: "NSbase+20%", 23: "NSbase-20%"}
            print(f" {sampling_names.get(action_id, 'Unknown')}")
        return action_id, action_dist.log_prob(action)
    def create_architecture_command(self, action_id):
        commands = {
            14: {'type': 'depth', 'direction': 'increase'},
            15: {'type': 'width', 'direction': 'increase'},
            16: {'type': 'depth', 'direction': 'decrease'},
            17: {'type': 'width', 'direction': 'decrease'}
        }
        return commands.get(action_id, None)

    def create_sampling_command(self, action_id):
        commands = {
            18: {'region': 'darcy', 'action': 'add_refined'},
            19: {'region': 'darcy', 'action': 'increase_base'},
            20: {'region': 'darcy', 'action': 'decrease_base'},
            21: {'region': 'ns', 'action': 'add_refined'},
            22: {'region': 'ns', 'action': 'increase_base'},
            23: {'region': 'ns', 'action': 'decrease_base'}
        }
        return commands.get(action_id, None)

    def update_policy(self):
        if len(self.episode_rewards) == 0:
            return
        discounted_rewards = []
        R = 0
        for r in reversed(self.episode_rewards):
            R = r + self.gamma * R
            discounted_rewards.insert(0, R)
        discounted_rewards = torch.tensor(discounted_rewards).to(device)
        if len(discounted_rewards) > 1:
            discounted_rewards = (discounted_rewards - discounted_rewards.mean()) / (discounted_rewards.std() + 1e-9)
        policy_loss = []
        for log_prob, G in zip(self.episode_log_probs, discounted_rewards):
            policy_loss.append(-log_prob * G)
        self.optimizer.zero_grad()
        policy_loss = torch.stack(policy_loss).sum()
        policy_loss.backward()
        self.optimizer.step()
        self.episode_rewards = []
        self.episode_log_probs = []

    def add_reward(self, reward):
        self.episode_rewards.append(reward)

    def add_log_prob(self, log_prob):
        self.episode_log_probs.append(log_prob)

    def print_statistics(self):
        if self.action_stats['total'] > 0:
            total = self.action_stats['total']
            
            print(f"  weight adjustment: {self.action_stats['weight']/total:.2%}")
            print(f"  architecture adjustment: {self.action_stats['arch']/total:.2%}")
            print(f"  sampling adjustment: {self.action_stats['sampling']/total:.2%}")


In [ ]:
"""Physics-informed model and training logic"""

class PhysicsInformedNN_YShape():
    def __init__(self, K, nu, g, alpha, s,
             interface_segments, velocity_bc_segments,
             X_ns_lhs_pool, X_darcy_lhs_pool,
             t_range=(0, 0.5), use_rl=False):

        self.k = K
        self.nu = nu
        self.g = g
        self.alpha = alpha
        self.s = s
        self.t_range = t_range
        self.interface_segments = interface_segments
        self.velocity_bc_segments = velocity_bc_segments

        self.k11 = K
        self.k22 = K
        self.d = 2
        self.trace = 2 * K

        self.X_ns_initial = X_ns_lhs_pool.copy()
        self.X_darcy_initial = X_darcy_lhs_pool.copy()
        self.N_f_ns_current = len(self.X_ns_initial)
        self.N_f_darcy_current = len(self.X_darcy_initial)
        self.N_ns_refined = 0
        self.N_darcy_refined = 0
        self.X_ns_refined_current = np.empty((0, 3))
        self.X_darcy_refined_current = np.empty((0, 3))

        self._generate_boundary_points()
        self.update_residual_points()

        self.arch_config = {
            'depth': 4, 'width': 64,
            'min_depth': 2, 'max_depth': 10,
            'min_units': 5, 'max_units': 256
        }
        self.build_network()

        self.optimizer = torch.optim.Adam(self.dnn.parameters(), lr=0.001, weight_decay=1e-5)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=500)

        self.weights = {
            'darcy': 10.0, 'ns': 10.0, 'interface': 10.0,
            'darcy_bc': 10.0, 'ns_bc': 10.0,
            'darcy_t0': 10.0, 'ns_t0': 10.0
        }
        self._clip_weights()

        self.use_rl = use_rl
        if self.use_rl:
            self.rl_controller = RLController(
                n_actions=24, state_dim=25,
                min_units=self.arch_config['min_units'],
                max_units=self.arch_config['max_units'],
                min_depth=self.arch_config['min_depth'],
                max_depth=self.arch_config['max_depth']
            )
            self.weight_adjust_interval = 200
            self.prev_losses = None
            self.initial_losses = None
            self.last_arch_adjust_step = -10000
            self.weight_adjustment_info = None
            self.sampling_adjustment_info = None

        self.iter = 0
        self.adjustment_info = None
        self.architecture_history = []
        self.record_architecture(0, "Initial")
        self.recent_losses = []
        self.loss_history = []
        self.historical_best_loss = float('inf')
        self.total_loss_history = []
        self.iteration_history = []
        self.consecutive_bad_steps = 0
        self.loss_explosion_threshold = 50.0
        self.sampling_history = []

    def _generate_boundary_points(self, n_per_segment=50, n_t=10):

        t_vals = np.linspace(self.t_range[0], self.t_range[1], n_t)

        interface_points = []
        interface_normals_f = []
        interface_tangents = []

        for name, seg in self.interface_segments.items():
            outward = seg['outward']

            for t_time in t_vals:
                s_vals = np.linspace(0, 1, n_per_segment)
                for s_val in s_vals:
                    if seg['type'] == 'cubic':

                        point = cubic_bezier(seg['p0'], seg['ctrl1'], seg['ctrl2'], seg['p3'], s_val)
                        tangent_raw = cubic_bezier_derivative(seg['p0'], seg['ctrl1'], seg['ctrl2'], seg['p3'], s_val)
                    elif seg['type'] == 'quadratic':

                        point = quadratic_bezier(seg['p0'], seg['ctrl'], seg['p2'], s_val)
                        tangent_raw = quadratic_bezier_derivative(seg['p0'], seg['ctrl'], seg['p2'], s_val)
                    else:

                        point = (1 - s_val) * seg['start'] + s_val * seg['end']
                        tangent_raw = seg['end'] - seg['start']

                    tangent_length = np.linalg.norm(tangent_raw)
                    if tangent_length > 1e-10:
                        tangent = tangent_raw / tangent_length
                    else:
                        tangent = np.array([1.0, 0.0])

                    if outward == 'right':
                        normal = np.array([tangent[1], -tangent[0]])
                    else:
                        normal = np.array([-tangent[1], tangent[0]])

                    interface_points.append([point[0], point[1], t_time])
                    interface_normals_f.append(normal)
                    interface_tangents.append(tangent)

        self.X_interface = np.array(interface_points)
        self.interface_nf = np.array(interface_normals_f)
        self.interface_np = -self.interface_nf
        self.interface_tau = np.array(interface_tangents)

        ns_bc_points = []
        ns_bc_values_u1 = []
        ns_bc_values_u2 = []

        for name, seg in self.velocity_bc_segments.items():
            p1, p2 = seg['start'], seg['end']
            u_val = seg['u']
            for t_time in t_vals:
                s_vals = np.linspace(0, 1, n_per_segment)
                for s_val in s_vals:
                    point = (1 - s_val) * p1 + s_val * p2
                    ns_bc_points.append([point[0], point[1], t_time])
                    ns_bc_values_u1.append(u_val[0])
                    ns_bc_values_u2.append(u_val[1])

        self.X_ns_bc = np.array(ns_bc_points)
        self.u1_bc = np.array(ns_bc_values_u1).reshape(-1, 1)
        self.u2_bc = np.array(ns_bc_values_u2).reshape(-1, 1)

        darcy_bc_points = []
        phi_bc_values = []

        for t_time in t_vals:

            x_vals = np.linspace(0, 1, n_per_segment)
            for x in x_vals:
                if not (0.3 <= x <= 0.5):
                    darcy_bc_points.append([x, 0, t_time])
                    phi_bc_values.append(0.0)

        for t_time in t_vals:

            x_vals = np.linspace(0, 1, n_per_segment)
            for x in x_vals:
                darcy_bc_points.append([x, 1, t_time])
                phi_bc_values.append(0.0)

        for t_time in t_vals:

            y_vals = np.linspace(0, 1, n_per_segment)
            for y in y_vals:
                if not (0.25 <= y <= 0.5):
                    darcy_bc_points.append([0, y, t_time])
                    phi_bc_values.append(0.0)

        for t_time in t_vals:

            y_vals = np.linspace(0, 1, n_per_segment)
            for y in y_vals:
                if not (0.5 <= y <= 0.75):
                    darcy_bc_points.append([1, y, t_time])
                    phi_bc_values.append(0.0)

        self.X_darcy_bc = np.array(darcy_bc_points)
        self.phi_bc = np.array(phi_bc_values).reshape(-1, 1)

        ns_t0_points = sample_in_y_shape(500, t_range=(0, 0))
        ns_t0_points[:, 2] = 0
        self.X_ns_t0 = ns_t0_points
        self.u1_t0 = np.zeros((len(ns_t0_points), 1))
        self.u2_t0 = np.zeros((len(ns_t0_points), 1))
        self.p_t0 = np.zeros((len(ns_t0_points), 1))

        darcy_t0_points = sample_in_porous_media(500, t_range=(0, 0))
        darcy_t0_points[:, 2] = 0
        self.X_darcy_t0 = darcy_t0_points
        self.phi_t0 = np.zeros((len(darcy_t0_points), 1))

        self.X_interface_fixed = self.X_interface.copy()
        self.X_ns_bc_fixed = self.X_ns_bc.copy()
        self.X_darcy_bc_fixed = self.X_darcy_bc.copy()
        self.X_ns_t0_fixed = self.X_ns_t0.copy()
        self.X_darcy_t0_fixed = self.X_darcy_t0.copy()


    def build_network(self):
        depth = self.arch_config['depth']
        width = self.arch_config['width']
        layers = []
        layers.append(nn.Linear(3, width))
        layers.append(nn.SiLU())
        for _ in range(depth):
            layers.append(nn.Linear(width, width))
            layers.append(nn.SiLU())
        layers.append(nn.Linear(width, 4))
        self.dnn = nn.Sequential(*layers).to(device)
        self.dnn.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def get_architecture_info(self):
        total_params = sum(p.numel() for p in self.dnn.parameters())
        return {'depth': self.arch_config['depth'], 'width': self.arch_config['width'], 'total_params': total_params}

    def record_architecture(self, step, action):
        info = self.get_architecture_info()
        info.update({'step': step, 'action': action})
        self.architecture_history.append(info)

    def adjust_architecture(self, command):
        if command is None:
            return False
        success = False
        if command['type'] == 'depth':
            if command['direction'] == 'increase':
                if self.arch_config['depth'] < self.arch_config['max_depth']:
                    self.arch_config['depth'] += 1
                    success = True
            else:
                if self.arch_config['depth'] > self.arch_config['min_depth']:
                    self.arch_config['depth'] -= 1
                    success = True
        elif command['type'] == 'width':
            current_width = self.arch_config['width']
            if command['direction'] == 'increase':
                new_width = min(int(current_width * 1.1), self.arch_config['max_units'])
                if new_width > current_width:
                    self.arch_config['width'] = new_width
                    success = True
            else:
                new_width = max(int(current_width * 0.9), self.arch_config['min_units'])
                if new_width < current_width:
                    self.arch_config['width'] = new_width
                    success = True
        if success:
            old_network = self.dnn
            self.build_network()
            self.migrate_weights(old_network, self.dnn)
            self.reset_optimizer()
            action_desc = f"{command['direction']} {command['type']}"
            self.record_architecture(self.iter, action_desc)
            print(f"architecture adjustment成功: {action_desc}")
            return True
        return False

    def migrate_weights(self, old_network, new_network):
        old_layers = [m for m in old_network.modules() if isinstance(m, nn.Linear)]
        new_layers = [m for m in new_network.modules() if isinstance(m, nn.Linear)]
        for old_layer, new_layer in zip(old_layers, new_layers):
            min_in = min(new_layer.in_features, old_layer.in_features)
            min_out = min(new_layer.out_features, old_layer.out_features)
            with torch.no_grad():
                new_layer.weight[:min_out, :min_in] = old_layer.weight[:min_out, :min_in]
                if new_layer.bias is not None and old_layer.bias is not None:
                    new_layer.bias[:min_out] = old_layer.bias[:min_out]

    def reset_optimizer(self):
        current_lr = self.optimizer.param_groups[0]['lr']
        self.optimizer = torch.optim.Adam(self.dnn.parameters(), lr=current_lr, weight_decay=1e-5)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='min', factor=0.5, patience=500)
        self.optimizer.state = defaultdict(dict)

    def _clip_weights(self):
        for k in self.weights:
            self.weights[k] = max(min(self.weights[k], 100.0), 1.0)

    def update_residual_points(self):

        self.ns_indices = np.random.choice(len(self.X_ns_initial),
                                            min(self.N_f_ns_current, len(self.X_ns_initial)), replace=False)
        selected_ns = self.X_ns_initial[self.ns_indices]

        self.darcy_indices = np.random.choice(len(self.X_darcy_initial),
                                               min(self.N_f_darcy_current, len(self.X_darcy_initial)), replace=False)
        selected_darcy = self.X_darcy_initial[self.darcy_indices]

        ns_components = [selected_ns]
        if len(self.X_ns_refined_current) > 0:
            ns_components.append(self.X_ns_refined_current)
        ns_components.append(self.X_ns_bc_fixed)
        ns_components.append(self.X_interface_fixed)
        ns_components.append(self.X_ns_t0_fixed)
        X_f_ns = np.vstack(ns_components)

        darcy_components = [selected_darcy]
        if len(self.X_darcy_refined_current) > 0:
            darcy_components.append(self.X_darcy_refined_current)
        darcy_components.append(self.X_darcy_bc_fixed)
        darcy_components.append(self.X_interface_fixed)
        darcy_components.append(self.X_darcy_t0_fixed)
        X_f_darcy = np.vstack(darcy_components)

        self.x_ns = torch.tensor(X_f_ns[:,0], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.y_ns = torch.tensor(X_f_ns[:,1], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.t_ns = torch.tensor(X_f_ns[:,2], dtype=torch.float32, requires_grad=True).to(device)[:,None]

        self.x_darcy = torch.tensor(X_f_darcy[:,0], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.y_darcy = torch.tensor(X_f_darcy[:,1], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.t_darcy = torch.tensor(X_f_darcy[:,2], dtype=torch.float32, requires_grad=True).to(device)[:,None]

        self.x_interface = torch.tensor(self.X_interface_fixed[:,0], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.y_interface = torch.tensor(self.X_interface_fixed[:,1], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.t_interface = torch.tensor(self.X_interface_fixed[:,2], dtype=torch.float32, requires_grad=True).to(device)[:,None]

        self.nf1 = torch.tensor(self.interface_nf[:,0], dtype=torch.float32).to(device)[:,None]
        self.nf2 = torch.tensor(self.interface_nf[:,1], dtype=torch.float32).to(device)[:,None]
        self.np1 = torch.tensor(self.interface_np[:,0], dtype=torch.float32).to(device)[:,None]
        self.np2 = torch.tensor(self.interface_np[:,1], dtype=torch.float32).to(device)[:,None]
        self.tau1 = torch.tensor(self.interface_tau[:,0], dtype=torch.float32).to(device)[:,None]
        self.tau2 = torch.tensor(self.interface_tau[:,1], dtype=torch.float32).to(device)[:,None]

        self.x_ns_bc = torch.tensor(self.X_ns_bc_fixed[:,0], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.y_ns_bc = torch.tensor(self.X_ns_bc_fixed[:,1], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.t_ns_bc = torch.tensor(self.X_ns_bc_fixed[:,2], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.u1_bc_tensor = torch.tensor(self.u1_bc, dtype=torch.float32).to(device)
        self.u2_bc_tensor = torch.tensor(self.u2_bc, dtype=torch.float32).to(device)

        self.x_darcy_bc = torch.tensor(self.X_darcy_bc_fixed[:,0], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.y_darcy_bc = torch.tensor(self.X_darcy_bc_fixed[:,1], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.t_darcy_bc = torch.tensor(self.X_darcy_bc_fixed[:,2], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.phi_bc_tensor = torch.tensor(self.phi_bc, dtype=torch.float32).to(device)

        self.x_ns_t0 = torch.tensor(self.X_ns_t0_fixed[:,0], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.y_ns_t0 = torch.tensor(self.X_ns_t0_fixed[:,1], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.t_ns_t0 = torch.tensor(self.X_ns_t0_fixed[:,2], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.u1_t0_tensor = torch.tensor(self.u1_t0, dtype=torch.float32).to(device)
        self.u2_t0_tensor = torch.tensor(self.u2_t0, dtype=torch.float32).to(device)
        self.p_t0_tensor = torch.tensor(self.p_t0, dtype=torch.float32).to(device)

        self.x_darcy_t0 = torch.tensor(self.X_darcy_t0_fixed[:,0], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.y_darcy_t0 = torch.tensor(self.X_darcy_t0_fixed[:,1], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.t_darcy_t0 = torch.tensor(self.X_darcy_t0_fixed[:,2], dtype=torch.float32, requires_grad=True).to(device)[:,None]
        self.phi_t0_tensor = torch.tensor(self.phi_t0, dtype=torch.float32).to(device)

    def adjust_sampling_points(self, region, action):
        if region == 'darcy':
            if action == 'add_refined':
                return self._add_refined_points('darcy')
            elif action == 'increase_base':
                return self._adjust_base_points('darcy', increase=True)
            elif action == 'decrease_base':
                return self._adjust_base_points('darcy', increase=False)
        elif region == 'ns':
            if action == 'add_refined':
                return self._add_refined_points('ns')
            elif action == 'increase_base':
                return self._adjust_base_points('ns', increase=True)
            elif action == 'decrease_base':
                return self._adjust_base_points('ns', increase=False)
        return False



    def _add_refined_points(self, region):

        from scipy.stats.qmc import LatinHypercube

        target_new_points = 1000

        results = self.physics_residuals(return_point_losses=True)

        if region == 'darcy':
            if self.N_darcy_refined >= self.rl_controller.max_refined_points:
                print(f" Darcyrefinedsamplinghas reached the upper limit({self.N_darcy_refined})")
                return False

            n_target = min(target_new_points, self.rl_controller.max_refined_points - self.N_darcy_refined)

            darcy_residuals = results[7].detach().cpu().numpy()
            n_base = self.N_f_darcy_current
            base_residuals = darcy_residuals[:n_base].flatten()

            current_darcy_points = self.X_darcy_initial[self.darcy_indices]

            threshold_idx = int(0.8 * len(base_residuals))
            high_loss_indices = np.argsort(base_residuals)[threshold_idx:]
            high_loss_coords = current_darcy_points[high_loss_indices]

            x_min, x_max = high_loss_coords[:, 0].min(), high_loss_coords[:, 0].max()
            y_min, y_max = high_loss_coords[:, 1].min(), high_loss_coords[:, 1].max()
            t_min, t_max = high_loss_coords[:, 2].min(), high_loss_coords[:, 2].max()

            x_range = x_max - x_min + 1e-6
            y_range = y_max - y_min + 1e-6
            t_range_val = t_max - t_min + 1e-6

            x_min = max(0, x_min - 0.05 * x_range)
            x_max = min(1, x_max + 0.05 * x_range)
            y_min = max(0, y_min - 0.05 * y_range)
            y_max = min(1, y_max + 0.05 * y_range)
            t_min = max(self.t_range[0], t_min - 0.05 * t_range_val)
            t_max = min(self.t_range[1], t_max + 0.05 * t_range_val)

            valid_points = []
            max_attempts = 10
            oversample_factor = 3

            for attempt in range(max_attempts):
                n_sample = int((n_target - len(valid_points)) * oversample_factor)
                n_sample = max(n_sample, 100)

                sampler = LatinHypercube(d=3)
                lhs_samples = sampler.random(n_sample)

                new_points = lhs_samples * [[x_max - x_min, y_max - y_min, t_max - t_min]] + [[x_min, y_min, t_min]]

                for p in new_points:
                    if not point_in_y_shape(p[0], p[1]):
                        valid_points.append(p)
                        if len(valid_points) >= n_target:
                            break

                if len(valid_points) >= n_target:
                    break

                oversample_factor *= 2

            valid_points = valid_points[:n_target]

            if len(valid_points) > 0:
                valid_points = np.array(valid_points)
                if len(self.X_darcy_refined_current) == 0:
                    self.X_darcy_refined_current = valid_points
                else:
                    self.X_darcy_refined_current = np.vstack([self.X_darcy_refined_current, valid_points])
                self.N_darcy_refined = len(self.X_darcy_refined_current)
              
                return True
            else:
               
                return False

        else:
            if self.N_ns_refined >= self.rl_controller.max_refined_points:
                print(f" NSrefinedsamplinghas reached the upper limit({self.N_ns_refined})")
                return False

            n_target = min(target_new_points, self.rl_controller.max_refined_points - self.N_ns_refined)

            ns_residuals = results[8].detach().cpu().numpy()
            n_base = self.N_f_ns_current
            base_residuals = ns_residuals[:n_base].flatten()

            current_ns_points = self.X_ns_initial[self.ns_indices]

            threshold_idx = int(0.8 * len(base_residuals))
            high_loss_indices = np.argsort(base_residuals)[threshold_idx:]
            high_loss_coords = current_ns_points[high_loss_indices]

            x_min, x_max = high_loss_coords[:, 0].min(), high_loss_coords[:, 0].max()
            y_min, y_max = high_loss_coords[:, 1].min(), high_loss_coords[:, 1].max()
            t_min, t_max = high_loss_coords[:, 2].min(), high_loss_coords[:, 2].max()

            x_range = x_max - x_min + 1e-6
            y_range = y_max - y_min + 1e-6
            t_range_val = t_max - t_min + 1e-6

            x_min = max(0, x_min - 0.05 * x_range)
            x_max = min(1, x_max + 0.05 * x_range)
            y_min = max(0, y_min - 0.05 * y_range)
            y_max = min(1, y_max + 0.05 * y_range)
            t_min = max(self.t_range[0], t_min - 0.05 * t_range_val)
            t_max = min(self.t_range[1], t_max + 0.05 * t_range_val)

            valid_points = []
            max_attempts = 10
            oversample_factor = 3

            for attempt in range(max_attempts):
                n_sample = int((n_target - len(valid_points)) * oversample_factor)
                n_sample = max(n_sample, 100)

                sampler = LatinHypercube(d=3)
                lhs_samples = sampler.random(n_sample)

                new_points = lhs_samples * [[x_max - x_min, y_max - y_min, t_max - t_min]] + [[x_min, y_min, t_min]]

                for p in new_points:
                    if point_in_y_shape(p[0], p[1]):
                        valid_points.append(p)
                        if len(valid_points) >= n_target:
                            break

                if len(valid_points) >= n_target:
                    break

                oversample_factor *= 2

            valid_points = valid_points[:n_target]

            if len(valid_points) > 0:
                valid_points = np.array(valid_points)
                if len(self.X_ns_refined_current) == 0:
                    self.X_ns_refined_current = valid_points
                else:
                    self.X_ns_refined_current = np.vstack([self.X_ns_refined_current, valid_points])
                self.N_ns_refined = len(self.X_ns_refined_current)
                
                return True
            else:
                
                return False

        return False
    def _adjust_base_points(self, region, increase):
        if region == 'darcy':
            current_n = self.N_f_darcy_current
            min_n = self.rl_controller.min_base_points
            max_n = len(self.X_darcy_initial)
            if increase:
                increment = min(int(0.2 * current_n), max_n - current_n)
                if increment > 0:
                    self.N_f_darcy_current = current_n + increment
         
                    return True
            else:
                decrement = min(int(0.2 * current_n), current_n - min_n)
                if decrement > 0:
                    self.N_f_darcy_current = current_n - decrement
                    
                    return True
        else:
            current_n = self.N_f_ns_current
            min_n = self.rl_controller.min_base_points
            max_n = len(self.X_ns_initial)
            if increase:
                increment = min(int(0.2 * current_n), max_n - current_n)
                if increment > 0:
                    self.N_f_ns_current = current_n + increment
                    
                    return True
            else:
                decrement = min(int(0.2 * current_n), current_n - min_n)
                if decrement > 0:
                    self.N_f_ns_current = current_n - decrement
                    
                    return True
        return False

    def update_loss_history(self, total_loss):
        self.recent_losses.append(total_loss)
        self.loss_history.append(total_loss)
        if len(self.recent_losses) > 100:
            self.recent_losses.pop(0)
        if len(self.loss_history) > 10000:
            self.loss_history.pop(0)

    def net_forward(self, x, y, t):
        inputs = torch.cat([x, y, t], dim=1)
        outputs = self.dnn(inputs)
        return outputs[:,0:1], outputs[:,1:2], outputs[:,2:3], outputs[:,3:4]

    def physics_residuals(self, return_point_losses=False):

        phi, _, _, _ = self.net_forward(self.x_darcy, self.y_darcy, self.t_darcy)
        dphi_t = torch.autograd.grad(phi, self.t_darcy, torch.ones_like(phi), create_graph=True)[0]
        dphi_x = torch.autograd.grad(phi, self.x_darcy, torch.ones_like(phi), create_graph=True)[0]
        dphi_y = torch.autograd.grad(phi, self.y_darcy, torch.ones_like(phi), create_graph=True)[0]
        up_x = self.k11 * dphi_x
        up_y = self.k22 * dphi_y
        div_up = torch.autograd.grad(up_x, self.x_darcy, grad_outputs=torch.ones_like(up_x), create_graph=True)[0] + \
                 torch.autograd.grad(up_y, self.y_darcy, grad_outputs=torch.ones_like(up_y), create_graph=True)[0]

        f_D = 0

        darcy_point_residuals = (self.s * dphi_t - (self.g /self.nu) * div_up - f_D)**2
        loss_darcy = torch.mean(darcy_point_residuals)

        _, u1, u2, p = self.net_forward(self.x_ns, self.y_ns, self.t_ns)

        du1dx = torch.autograd.grad(u1, self.x_ns, torch.ones_like(u1), create_graph=True)[0]
        du1dy = torch.autograd.grad(u1, self.y_ns, torch.ones_like(u1), create_graph=True)[0]
        du2dx = torch.autograd.grad(u2, self.x_ns, torch.ones_like(u2), create_graph=True)[0]
        du2dy = torch.autograd.grad(u2, self.y_ns, torch.ones_like(u2), create_graph=True)[0]

        d2u1dx2 = torch.autograd.grad(du1dx, self.x_ns, grad_outputs=torch.ones_like(du1dx), create_graph=True)[0]
        d2u1dy2 = torch.autograd.grad(du1dy, self.y_ns, grad_outputs=torch.ones_like(du1dy), create_graph=True)[0]
        d2u2dx2 = torch.autograd.grad(du2dx, self.x_ns, grad_outputs=torch.ones_like(du2dx), create_graph=True)[0]
        d2u2dy2 = torch.autograd.grad(du2dy, self.y_ns, grad_outputs=torch.ones_like(du2dy), create_graph=True)[0]

        du1dt = torch.autograd.grad(u1, self.t_ns, grad_outputs=torch.ones_like(u1), create_graph=True)[0]
        du2dt = torch.autograd.grad(u2, self.t_ns, grad_outputs=torch.ones_like(u2), create_graph=True)[0]

        t11 = 2*self.nu*du1dx-p
        t12 = self.nu*(du1dy+du2dx)
        t21 = self.nu*(du1dy+du2dx)
        t22 = 2*self.nu*du2dy-p
        dt11dx = torch.autograd.grad(t11, self.x_ns, torch.ones_like(t11), create_graph=True)[0]
        dt12dy = torch.autograd.grad(t12, self.y_ns, torch.ones_like(t12), create_graph=True)[0]
        dt21dx = torch.autograd.grad(t21, self.x_ns, torch.ones_like(t21), create_graph=True)[0]
        dt22dy = torch.autograd.grad(t22, self.y_ns, torch.ones_like(t22), create_graph=True)[0]

        f1=0
        f2=0

        loss_1 = du1dt + (u1*du1dx+u2*du1dy) - (dt11dx+dt12dy) - f1
        loss_2 = du2dt + (u1*du2dx+u2*du2dy) - (dt21dx+dt22dy) - f2
        loss_3 = du1dx + du2dy

        ns_point_residuals = loss_1**2 + loss_2**2 + loss_3**2
        loss_ns = torch.mean(ns_point_residuals)

        phi_inter, u1_inter, u2_inter, p_inter = self.net_forward(self.x_interface, self.y_interface, self.t_interface)
        dphi_x = torch.autograd.grad(phi_inter, self.x_interface, grad_outputs=torch.ones_like(phi_inter), create_graph=True)[0]
        dphi_y = torch.autograd.grad(phi_inter, self.y_interface, grad_outputs=torch.ones_like(phi_inter), create_graph=True)[0]
        du1dx = torch.autograd.grad(u1_inter, self.x_interface, grad_outputs=torch.ones_like(u1_inter), create_graph=True)[0]
        du1dy = torch.autograd.grad(u1_inter, self.y_interface, grad_outputs=torch.ones_like(u1_inter), create_graph=True)[0]
        du2dx = torch.autograd.grad(u2_inter, self.x_interface, grad_outputs=torch.ones_like(u2_inter), create_graph=True)[0]
        du2dy = torch.autograd.grad(u2_inter, self.y_interface, grad_outputs=torch.ones_like(u2_inter), create_graph=True)[0]
        t11 = 2*self.nu*du1dx-p_inter
        t12 = self.nu*(du1dy+du2dx)
        t21 = self.nu*(du1dy+du2dx)
        t22 = 2*self.nu*du2dy-p_inter

        loss_1 = u1_inter * self.nf1 + u2_inter * self.nf2 - (self.g / self.nu) * (self.k11 * dphi_x * self.np1 + self.k22 * dphi_y * self.np2)
        loss_2 = -self.nf1 * (t11*self.nf1 + t12*self.nf2) -self.nf2 * (t21*self.nf1 + t22*self.nf2) - self.g*phi_inter

        loss_3 = -self.tau1 * (t11*self.nf1 + t12*self.nf2) -self.tau2 * (t21*self.nf1 + t22*self.nf2) - (self.alpha * self.nu * (self.d ** 0.5) / (self.trace ** 0.5))*(self.tau1 * (u1_inter + (self.g / self.nu) * self.k11 * dphi_x) + self.tau2 * (u2_inter + (self.g / self.nu) * self.k22 * dphi_y))

        loss_interface = torch.mean((loss_1)**2 + (loss_2)**2 + (loss_3)**2)

        phi_bc_pred, _, _, _ = self.net_forward(self.x_darcy_bc, self.y_darcy_bc, self.t_darcy_bc)
        loss_darcy_bc = torch.mean((phi_bc_pred - self.phi_bc_tensor)**2)

        _, u1_bc_pred, u2_bc_pred, _ = self.net_forward(self.x_ns_bc, self.y_ns_bc, self.t_ns_bc)
        loss_ns_bc = torch.mean((u1_bc_pred - self.u1_bc_tensor)**2) + torch.mean((u2_bc_pred - self.u2_bc_tensor)**2)

        phi_t0_pred, _, _, _ = self.net_forward(self.x_darcy_t0, self.y_darcy_t0, self.t_darcy_t0)
        loss_darcy_t0 = torch.mean((phi_t0_pred - self.phi_t0_tensor)**2)

        _, u1_t0_pred, u2_t0_pred, p_t0_pred = self.net_forward(self.x_ns_t0, self.y_ns_t0, self.t_ns_t0)
        loss_ns_t0 = torch.mean((u1_t0_pred - self.u1_t0_tensor)**2) + torch.mean((u2_t0_pred - self.u2_t0_tensor)**2) + torch.mean((p_t0_pred - self.p_t0_tensor)**2)

        if return_point_losses:
            return (loss_darcy, loss_ns, loss_interface, loss_darcy_bc, loss_ns_bc, loss_darcy_t0, loss_ns_t0, darcy_point_residuals, ns_point_residuals)
        else:
            return loss_darcy, loss_ns, loss_interface, loss_darcy_bc, loss_ns_bc, loss_darcy_t0, loss_ns_t0

    def loss_func(self, training=True):
        if training:
            self.optimizer.zero_grad()
        losses = self.physics_residuals(return_point_losses=False)
        self.current_losses = losses
        if self.initial_losses is None:
            self.initial_losses = [l.item() for l in losses]

        current_total_unweighted = sum([l.item() for l in losses])
        if current_total_unweighted < self.historical_best_loss:
            self.historical_best_loss = current_total_unweighted

        if self.use_rl and (self.iter % self.weight_adjust_interval == 0):
            arch_info = self.get_architecture_info()
            current_losses = [l.item() for l in losses]

            state = self.rl_controller.get_enhanced_state(
                current_losses, arch_info['depth'], arch_info['width'], self.loss_history,
                self.N_f_darcy_current, self.N_f_ns_current, self.N_darcy_refined, self.N_ns_refined,
                historical_best_loss=self.historical_best_loss,
                individual_losses=current_losses
            )
            action_id, log_prob = self.rl_controller.select_action(state)

            if action_id < 14:
                action_map = {0: +10, 1: -10}
                adjust_factor = action_map[action_id % 2]
                target_loss_idx = action_id // 2
                loss_keys = ['darcy', 'ns', 'interface', 'darcy_bc', 'ns_bc', 'darcy_t0', 'ns_t0']
                selected_key = loss_keys[target_loss_idx]
                self.weights[selected_key] += adjust_factor
                self._clip_weights()
                self.rl_controller.add_log_prob(log_prob)
                self.weight_adjustment_info = {
                    'step': self.iter,
                    'pre_losses': current_losses.copy(),
                    'action_id': action_id
                }
            elif action_id < 18:
                command = self.rl_controller.create_architecture_command(action_id)
                self.adjustment_info = {'step': self.iter, 'pre_losses': current_losses, 'arch_action': action_id, 'log_prob': log_prob, 'command': command}
                self.rl_controller.add_log_prob(log_prob)
                if self.adjust_architecture(command):
                    self.last_arch_adjust_step = self.iter
                    self.adjustment_info['action_desc'] = f"{command['direction']} {command['type']}"
            else:
                command = self.rl_controller.create_sampling_command(action_id)
                if command:
                    self.rl_controller.add_log_prob(log_prob)
                    pre_losses = [l.item() for l in losses]
                    success = self.adjust_sampling_points(command['region'], command['action'])
                    if success:
                        self.update_residual_points()
                        self.sampling_adjustment_info = {
                            'step': self.iter,
                            'pre_losses': pre_losses,
                            'command': command
                        }

        if self.adjustment_info and (self.iter - self.adjustment_info['step']) >= 200:
            current_losses = [l.item() for l in losses]
            pre_losses = self.adjustment_info['pre_losses']
            arch_info = self.get_architecture_info()
            arch_reward = calculate_arch_reward(pre_losses, current_losses, arch_info['total_params'])
            self.rl_controller.add_reward(arch_reward)
            self.adjustment_info = None

        if self.weight_adjustment_info and (self.iter - self.weight_adjustment_info['step']) >= 200:
            current_losses_val = [l.item() for l in losses]
            pre_losses_val = self.weight_adjustment_info['pre_losses']
            weight_reward = calculate_reward(current_losses_val, pre_losses_val, self.initial_losses)
            self.rl_controller.add_reward(weight_reward)
            self.weight_adjustment_info = None

        if self.sampling_adjustment_info and (self.iter - self.sampling_adjustment_info['step']) >= 200:
            current_losses_val = [l.item() for l in losses]
            pre_losses_val = self.sampling_adjustment_info['pre_losses']
            sampling_reward = calculate_reward(current_losses_val, pre_losses_val, self.initial_losses)
            self.rl_controller.add_reward(sampling_reward)
            self.sampling_adjustment_info = None

        total_loss = sum([
            self.weights['darcy'] * losses[0], self.weights['ns'] * losses[1],
            self.weights['interface'] * losses[2], self.weights['darcy_bc'] * losses[3],
            self.weights['ns_bc'] * losses[4], self.weights['darcy_t0'] * losses[5],
            self.weights['ns_t0'] * losses[6]
        ])

        self.update_loss_history(total_loss.item())
        if self.iter % 10 == 0:
            self.total_loss_history.append(total_loss.item())
            self.iteration_history.append(self.iter)

        self._clip_weights()
        self.iter += 1

        if self.iter % 100 == 0:
            self.scheduler.step(total_loss.item())

        if self.use_rl and (self.iter % 500 == 0) and (len(self.rl_controller.episode_rewards) > 0):
            self.rl_controller.episode_rewards = [r * 0.1 for r in self.rl_controller.episode_rewards]
            self.rl_controller.update_policy()

        if self.iter % 100 == 0:
            arch_info = self.get_architecture_info()
            total_darcy = self.N_f_darcy_current + self.N_darcy_refined
            total_ns = self.N_f_ns_current + self.N_ns_refined

            print(
                f'Iter {self.iter} | Total Loss: {total_loss.item():.2e} | '
                f'Depth: {arch_info["depth"]}, Width: {arch_info["width"]} | '
                f'Darcysampling points: {self.N_f_darcy_current}+{self.N_darcy_refined}={self.N_f_darcy_current + self.N_darcy_refined}, '
                f'NSsampling points: {self.N_f_ns_current}+{self.N_ns_refined}={self.N_f_ns_current + self.N_ns_refined} | '
                f'LR: {self.optimizer.param_groups[0]["lr"]:.1e} | '
                f'Darcy: {losses[0].item():.2e} (w={self.weights["darcy"]:.2f}), '
                f'NS: {losses[1].item():.2e} (w={self.weights["ns"]:.2f}), '
                f'Interface: {losses[2].item():.2e} (w={self.weights["interface"]:.2f}), '
                f'Darcy_BC: {losses[3].item():.2e} (w={self.weights["darcy_bc"]:.2f}), '
                f'NS_BC: {losses[4].item():.2e} (w={self.weights["ns_bc"]:.2f}), '
                f'Darcy_T0: {losses[5].item():.2e} (w={self.weights["darcy_t0"]:.2f}), '
                f'NS_T0: {losses[6].item():.2e} (w={self.weights["ns_t0"]:.2f})'
            )

            if self.use_rl:
                self.rl_controller.print_statistics()

        return total_loss

    def train(self, stage1_epochs=20000, stage2_epochs=10000):
        print("=" * 60)
        print("Start two-stage training")
        print("=" * 60)

        training_start_time = time.time()

        initial_config = {
            'depth': self.arch_config['depth'],
            'width': self.arch_config['width'],
            'weights': self.weights.copy(),
            'N_darcy_base': self.N_f_darcy_current,
            'N_ns_base': self.N_f_ns_current,
            'N_darcy_refined': self.N_darcy_refined,
            'N_ns_refined': self.N_ns_refined
        }

        print(f"\n Stage 1:动态调整训练 ({stage1_epochs}步)")
        print("   - Enablearchitecture adjustment")
        print("   - Enableweight adjustment")
        print("   - Enablerefinedsampling adjustment")
        print("   - Enablereinforcement learning")

        stage1_start_time = time.time()
        self.dnn.train()
        stage1_losses = []

        for epoch in range(stage1_epochs):
            self.optimizer.zero_grad()
            loss = self.loss_func(training=True)
            loss.backward()
            self.optimizer.step()

            stage1_losses.append(loss.item())

            if epoch % 1000 == 0:
                arch_info = self.get_architecture_info()
                elapsed_time = time.time() - stage1_start_time
                total_darcy = self.N_f_darcy_current + self.N_darcy_refined
                total_ns = self.N_f_ns_current + self.N_ns_refined

                current_losses = self.current_losses

                print(
                    f'\nStage 1 -  {epoch}/{stage1_epochs} | Total Loss: {loss.item():.2e} | '
                    f'Depth: {arch_info["depth"]}, Width: {arch_info["width"]} | '
                    f'Darcysampling points: {self.N_f_darcy_current}+{self.N_darcy_refined}={total_darcy}, '
                    f'NSsampling points: {self.N_f_ns_current}+{self.N_ns_refined}={total_ns} | '
                    f'LR: {self.optimizer.param_groups[0]["lr"]:.1e} | elapsed time: {elapsed_time:.1f}s | '
                    f'Darcy: {current_losses[0].item():.2e} (w={self.weights["darcy"]:.2f}), '
                    f'NS: {current_losses[1].item():.2e} (w={self.weights["ns"]:.2f}), '
                    f'Interface: {current_losses[2].item():.2e} (w={self.weights["interface"]:.2f}), '
                    f'Darcy_BC: {current_losses[3].item():.2e} (w={self.weights["darcy_bc"]:.2f}), '
                    f'NS_BC: {current_losses[4].item():.2e} (w={self.weights["ns_bc"]:.2f}), '
                    f'Darcy_T0: {current_losses[5].item():.2e} (w={self.weights["darcy_t0"]:.2f}), '
                    f'NS_T0: {current_losses[6].item():.2e} (w={self.weights["ns_t0"]:.2f})'
                )

        stage1_end_time = time.time()
        stage1_duration = stage1_end_time - stage1_start_time

        final_iter_stage1 = self.iter - 1
        if final_iter_stage1 not in self.iteration_history:
            if len(stage1_losses) > 0:
                self.total_loss_history.append(stage1_losses[-1])
                self.iteration_history.append(final_iter_stage1)
                print(f" Stage 1Supplement the final step record: iter={final_iter_stage1}, loss={stage1_losses[-1]:.2e}")

        stage1_final_config = {
            'depth': self.arch_config['depth'],
            'width': self.arch_config['width'],
            'weights': self.weights.copy(),
            'N_darcy_base': self.N_f_darcy_current,
            'N_ns_base': self.N_f_ns_current,
            'N_darcy_refined': self.N_darcy_refined,
            'N_ns_refined': self.N_ns_refined,
            'final_loss': stage1_losses[-1],
            'training_time': stage1_duration
        }

        print(f"\n Stage 1completed (elapsed time: {stage1_duration:.1f}s, {stage1_duration/60:.1f}min):")
        print(f"   Initial configuration: depth={initial_config['depth']}, width={initial_config['width']}")
        print(f"   Final configuration: depth={stage1_final_config['depth']}, width={stage1_final_config['width']}")
        print(f"   Darcysampling: base{stage1_final_config['N_darcy_base']} + refined{stage1_final_config['N_darcy_refined']} = {stage1_final_config['N_darcy_base'] + stage1_final_config['N_darcy_refined']}")
        print(f"   NSsampling: base{stage1_final_config['N_ns_base']} + refined{stage1_final_config['N_ns_refined']} = {stage1_final_config['N_ns_base'] + stage1_final_config['N_ns_refined']}")
        print(f"   Finalloss: {stage1_final_config['final_loss']:.2e}")

        print(f"\n Stage 2({stage2_epochs})")
        print("   - Disablearchitecture adjustment")
        print("   - Disableweight adjustment")
        print("   - Disablesampling adjustment")
        print("   - Disablereinforcement learning")

        stage2_start_time = time.time()

        original_use_rl = self.use_rl
        self.use_rl = False

        stage2_losses = []

        for epoch in range(stage2_epochs):
            self.optimizer.zero_grad()

            losses = self.physics_residuals()
            total_loss = sum([
                self.weights['darcy'] * losses[0],
                self.weights['ns'] * losses[1],
                self.weights['interface'] * losses[2],
                self.weights['darcy_bc'] * losses[3],
                self.weights['ns_bc'] * losses[4],
                self.weights['darcy_t0'] * losses[5],
                self.weights['ns_t0'] * losses[6]
            ])

            self.update_loss_history(total_loss.item())
            if self.iter % 10 == 0:
                self.total_loss_history.append(total_loss.item())
                self.iteration_history.append(self.iter)

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(self.dnn.parameters(), max_norm=1.0)

            self.optimizer.step()

            current_loss = total_loss.item()
            stage2_losses.append(current_loss)

            if epoch % 100 == 0:
                self.scheduler.step(current_loss)

            if epoch % 1000 == 0:
                elapsed_time = time.time() - stage2_start_time
                arch_info = self.get_architecture_info()
                total_darcy = self.N_f_darcy_current + self.N_darcy_refined
                total_ns = self.N_f_ns_current + self.N_ns_refined

                print(
                    f'\nStage 2 -  {epoch}/{stage2_epochs} | Total Loss: {current_loss:.2e} | '
                    f'Depth: {arch_info["depth"]}, Width: {arch_info["width"]} | '
                    f'Darcysampling points: {self.N_f_darcy_current}+{self.N_darcy_refined}={total_darcy}, '
                    f'NSsampling points: {self.N_f_ns_current}+{self.N_ns_refined}={total_ns} | '
                    f'LR: {self.optimizer.param_groups[0]["lr"]:.1e} | elapsed time: {elapsed_time:.1f}s | '
                    f'Darcy: {losses[0].item():.2e} (w={self.weights["darcy"]:.2f}), '
                    f'NS: {losses[1].item():.2e} (w={self.weights["ns"]:.2f}), '
                    f'Interface: {losses[2].item():.2e} (w={self.weights["interface"]:.2f}), '
                    f'Darcy_BC: {losses[3].item():.2e} (w={self.weights["darcy_bc"]:.2f}), '
                    f'NS_BC: {losses[4].item():.2e} (w={self.weights["ns_bc"]:.2f}), '
                    f'Darcy_T0: {losses[5].item():.2e} (w={self.weights["darcy_t0"]:.2f}), '
                    f'NS_T0: {losses[6].item():.2e} (w={self.weights["ns_t0"]:.2f})'
                )

            self.iter += 1

        stage2_end_time = time.time()
        stage2_duration = stage2_end_time - stage2_start_time

        final_iter_stage2 = self.iter - 1
        if final_iter_stage2 not in self.iteration_history:
            if len(stage2_losses) > 0:
                self.total_loss_history.append(stage2_losses[-1])
                self.iteration_history.append(final_iter_stage2)
                print(f" Stage 2Supplement the final step record: iter={final_iter_stage2}, loss={stage2_losses[-1]:.2e}")

        self.use_rl = original_use_rl

        total_training_time = time.time() - training_start_time

        print(f"\n Two-stage training completed:")
        print(f"   Stage 1elapsed time: {stage1_duration:.1f}s ({stage1_duration/60:.1f}min)")
        print(f"   Stage 2elapsed time: {stage2_duration:.1f}s ({stage2_duration/60:.1f}min)")
        print(f"   Total training time: {total_training_time:.1f}s ({total_training_time/60:.1f}min)")
        print(f"   Stage 1Finalloss: {stage1_final_config['final_loss']:.2e}")
        print(f"   Stage 2Finalloss: {stage2_losses[-1]:.2e}")
        print(f"   Finalarchitecture: depth={self.arch_config['depth']}, width={self.arch_config['width']}")

        return {
            'stage1_losses': stage1_losses,
            'stage2_losses': stage2_losses,
            'stage1_config': stage1_final_config,
            'stage1_time': stage1_duration,
            'stage2_time': stage2_duration,
            'total_time': total_training_time
        }

    def predict_with_velocity(self, XYT):

        X = torch.tensor(XYT[:, 0:1], dtype=torch.float32, requires_grad=True).to(device)
        Y = torch.tensor(XYT[:, 1:2], dtype=torch.float32, requires_grad=True).to(device)
        T = torch.tensor(XYT[:, 2:3], dtype=torch.float32).to(device)

        phi, u1, u2, p = self.net_forward(X, Y, T)

        dphi_x = torch.autograd.grad(phi, X, torch.ones_like(phi), retain_graph=True, create_graph=False)[0]
        dphi_y = torch.autograd.grad(phi, Y, torch.ones_like(phi), retain_graph=False, create_graph=False)[0]

        up_x = -self.k * dphi_x
        up_y = -self.k * dphi_y

        return (phi.detach().cpu().numpy(),
                u1.detach().cpu().numpy(),
                u2.detach().cpu().numpy(),
                p.detach().cpu().numpy(),
                up_x.detach().cpu().numpy(),
                up_y.detach().cpu().numpy())

In [ ]:
"""Physical parameters and data preparation"""



K = 1.0
nu = 1.0
g = 1.0
alpha = 1.0
S0 = 1.0
T_final = 1.0
t_range = (0, T_final)


X_ns_lhs_pool = sample_lhs_in_y_shape(5000, t_range=t_range)
X_darcy_lhs_pool = sample_lhs_in_porous_media(5000, t_range=t_range)
print(f" NSpool: {X_ns_lhs_pool.shape}, Darcypool: {X_darcy_lhs_pool.shape}")



In [ ]:
"""Geometry and boundary definitions"""

model = PhysicsInformedNN_YShape(
    K=K, nu=nu, g=g, alpha=alpha, s=S0,
    interface_segments=interface_segments,
    velocity_bc_segments=velocity_bc_segments,
    X_ns_lhs_pool=X_ns_lhs_pool,
    X_darcy_lhs_pool=X_darcy_lhs_pool,
    t_range=t_range, use_rl=True
)



In [ ]:
"""Model training and summary"""

print("\nStart training...")
training_results = model.train(stage1_epochs=11000, stage2_epochs=10000)


In [ ]:
"""Prediction and timing summary"""



def points_in_polygon_vectorized(points_x, points_y, polygon):
    n = len(polygon)
    inside = np.zeros(len(points_x), dtype=bool)

    j = n - 1
    for i in range(n):
        xi, yi = polygon[i]
        xj, yj = polygon[j]

        cond1 = (yi > points_y) != (yj > points_y)
        cond2 = points_x < (xj - xi) * (points_y - yi) / (yj - yi + 1e-12) + xi

        inside ^= (cond1 & cond2)
        j = i

    return inside

def compute_mask_fast(X_mesh, Y_mesh, polygon):

    x_flat = X_mesh.flatten()
    y_flat = Y_mesh.flatten()
    inside_flat = points_in_polygon_vectorized(x_flat, y_flat, polygon)
    return inside_flat.reshape(X_mesh.shape)

visualization_times = [0.2, 0.5, 0.6, 0.8, 1.0]

n_grid = 1000

x_grid = np.linspace(0, 1, n_grid)
y_grid = np.linspace(0, 1, n_grid)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
X_flat = X_mesh.flatten()
Y_flat = Y_mesh.flatten()


t_mask_start = time.time()
mask_ns = compute_mask_fast(X_mesh, Y_mesh, y_shape_curved_polygon)
mask_darcy = ~mask_ns
print(f" Mask computation completed, elapsed time: {time.time() - t_mask_start:.2f}s")

save_path = './open_source_assets/curve3-multi-time-1.1w/'
os.makedirs(save_path, exist_ok=True)

def get_curved_boundary_for_plot(n_points_per_segment=50):
    boundary = generate_y_shape_boundary_polygon(n_points_per_segment)
    boundary = np.vstack([boundary, boundary[0]])
    return boundary

y_vertices = get_curved_boundary_for_plot(n_points_per_segment=50)

velocity_dataset = {
    'x': X_flat,
    'y': Y_flat,
    'mask_ns': mask_ns.flatten(),
    'mask_darcy': mask_darcy.flatten(),
    'times': [],
    'U_combined': [],
    'V_combined': [],
    'velocity_magnitude': [],
    'phi': [],
    'u1': [],
    'u2': [],
    'p': [],
    'up_x': [],
    'up_y': []
}

for t_target in visualization_times:
    print(f"\n{'='*40}")
    print(f"Processing time point t = {t_target:.2f}")
    print(f"{'='*40}")

    T_flat = np.full_like(X_flat, t_target)
    XYT_pred = np.column_stack([X_flat, Y_flat, T_flat])


    t_pred_start = time.time()
    phi_pred, u1_pred, u2_pred, p_pred, up_x_pred, up_y_pred = model.predict_with_velocity(XYT_pred)
    print(f" Prediction completed, elapsed time: {time.time() - t_pred_start:.2f}s")

    u1_grid = u1_pred.reshape(n_grid, n_grid)
    u2_grid = u2_pred.reshape(n_grid, n_grid)
    up_x_grid = up_x_pred.reshape(n_grid, n_grid)
    up_y_grid = up_y_pred.reshape(n_grid, n_grid)

    U_combined = np.zeros_like(u1_grid)
    V_combined = np.zeros_like(u2_grid)
    U_combined[mask_ns] = u1_grid[mask_ns]
    V_combined[mask_ns] = u2_grid[mask_ns]
    U_combined[mask_darcy] = up_x_grid[mask_darcy]
    V_combined[mask_darcy] = up_y_grid[mask_darcy]
    velocity_magnitude = np.sqrt(U_combined**2 + V_combined**2)


    fig, ax = plt.subplots(figsize=(12, 10))

    im = ax.pcolormesh(X_mesh, Y_mesh, velocity_magnitude, cmap='jet', shading='auto')
    plt.colorbar(im, ax=ax, label='Velocity Magnitude')

    ax.plot(y_vertices[:, 0], y_vertices[:, 1], 'k-', linewidth=2)

    ax.set_xlabel('x', fontsize=14)
    ax.set_ylabel('y', fontsize=14)

    ax.set_aspect('equal')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'y_shape_velocity_field_t{t_target:.2f}_k1.png'),
                dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(save_path, f'y_shape_velocity_field_t{t_target:.2f}_k1.eps'),
                format='eps', bbox_inches='tight')
    print(f" velocity-field figure t={t_target:.2f} saved")
    plt.close()

   
    fig, ax = plt.subplots(figsize=(12, 10))

    im = ax.pcolormesh(X_mesh, Y_mesh, velocity_magnitude, cmap='jet', shading='auto')
    plt.colorbar(im, ax=ax, label='Velocity Magnitude')
    ax.plot(y_vertices[:, 0], y_vertices[:, 1], 'k-', linewidth=2)

    try:
        ax.streamplot(X_mesh, Y_mesh, U_combined, V_combined,
                     color='white', linewidth=0.8, density=1.5)
        print(f" streamline plot succeeded")
    except Exception as e:
        print(f" streamline plot failed: {e}")

    ax.set_xlabel('x', fontsize=14)
    ax.set_ylabel('y', fontsize=14)

    ax.set_aspect('equal')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'y_shape_velocity_streamlines_t{t_target:.2f}_k1.png'),
                dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(save_path, f'y_shape_velocity_streamlines_t{t_target:.2f}_k1.eps'),
                format='eps', bbox_inches='tight')
    print(f" streamline figure t={t_target:.2f} saved")
    plt.close()


plt.figure(figsize=(12, 6))
plt.semilogy(model.iteration_history, model.total_loss_history, 'b-', linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Total Loss (log scale)', fontsize=12)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(save_path, 'total_loss_curve.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(save_path, 'total_loss_curve.eps'), format='eps', bbox_inches='tight')
print(f" losscurvesaved")
plt.close()
